# Hafta 3 - Ödev: Multi-threaded ve CUDA destekli gerçekleme

Görev 1-5 (kontrast germe, histogram eşitleme, CLAHE, 3x3 ortalama, 5x5 medyan) hem çoklu thread'li CPU hem de CUDA çekirdekleriyle çalıştırılır, süreler karşılaştırılır ve GPU sonucunun CPU ile birebir aynı olduğu kontrol edilir.

> Önce: *Çalışma zamanı → Çalışma zamanı türünü değiştir → T4 GPU* seçin.

## 1. GPU kontrolü ve OpenCV (sadece resim okuma/yazma için) kurulumu

In [ ]:
!nvidia-smi --query-gpu=name,compute_cap --format=csv
!nvcc --version | tail -1
!apt-get -qq install -y libopencv-dev > /dev/null && pkg-config --modversion opencv4

## 2. Kaynak dosyalar

In [ ]:
!mkdir -p /content/hafta3/odev /content/hafta3/goruntuler

In [ ]:
%%writefile /content/hafta3/odev/islemler.hpp
// Hafta 3 - Ödev: görevlerin çoklu iş parçacıklı (multi-threaded) CPU sürümleri
//
// Her işlem, kullanılacak thread sayısını parametre olarak alır. Görüntü
// satır bantlarına bölünür ve her bandı ayrı bir std::thread işler.
// Thread'ler farklı satırlara yazdığı için kilit (mutex) gerekmez; ortak
// toplama gereken yerlerde (min/max, histogram) her thread kendi kısmi
// sonucunu üretir, thread'ler bitince bu sonuçlar birleştirilir.
//
// Bu dosya hem main_mt.cpp hem de main_cuda.cu tarafından (CPU referansı
// olarak) kullanılır.

#pragma once

#include <opencv2/opencv.hpp>

#include <algorithm>
#include <cmath>
#include <thread>
#include <vector>

namespace islem {

// [0, n) aralığını 'threadSayisi' parçaya böler; her parça için
// fonk(parcaNo, bas, bit) ayrı bir thread'de çalıştırılır.
template <typename F>
void paralel(int n, int threadSayisi, F fonk) {
    threadSayisi = std::max(1, std::min(threadSayisi, n));
    if (threadSayisi == 1) {
        fonk(0, 0, n);
        return;
    }
    std::vector<std::thread> threadler;
    for (int t = 0; t < threadSayisi; ++t) {
        int bas = static_cast<int>(static_cast<long long>(n) * t / threadSayisi);
        int bit = static_cast<int>(static_cast<long long>(n) * (t + 1) / threadSayisi);
        threadler.emplace_back(fonk, t, bas, bit);
    }
    for (auto& th : threadler) th.join();
}

// Tablo (LUT) uygulama: cikis(y,x) = tablo[giris(y,x)]
inline cv::Mat tabloUygula(const cv::Mat& gri, const uchar tablo[256], int threadSayisi) {
    cv::Mat cikis(gri.size(), CV_8UC1);
    paralel(gri.rows, threadSayisi, [&](int, int bas, int bit) {
        for (int y = bas; y < bit; ++y) {
            const uchar* g = gri.ptr<uchar>(y);
            uchar* c = cikis.ptr<uchar>(y);
            for (int x = 0; x < gri.cols; ++x) c[x] = tablo[g[x]];
        }
    });
    return cikis;
}

// ---------------------------------------------------------------------------
// Görev 1: min/max ile doğrusal kontrast germe
// ---------------------------------------------------------------------------
inline void kontrastTablosu(int enKucuk, int enBuyuk, uchar tablo[256]) {
    int aralik = std::max(enBuyuk - enKucuk, 1);
    for (int v = 0; v < 256; ++v) {
        int fark = std::clamp(v - enKucuk, 0, aralik);
        tablo[v] = static_cast<uchar>((fark * 255 * 2 + aralik) / (2 * aralik));
    }
}

inline cv::Mat kontrastGer(const cv::Mat& gri, int threadSayisi) {
    std::vector<int> mins(threadSayisi, 255), maxs(threadSayisi, 0);
    paralel(gri.rows, threadSayisi, [&](int t, int bas, int bit) {
        int mn = 255, mx = 0;
        for (int y = bas; y < bit; ++y) {
            const uchar* s = gri.ptr<uchar>(y);
            for (int x = 0; x < gri.cols; ++x) {
                mn = std::min<int>(mn, s[x]);
                mx = std::max<int>(mx, s[x]);
            }
        }
        mins[t] = mn;
        maxs[t] = mx;
    });
    int enKucuk = *std::min_element(mins.begin(), mins.end());
    int enBuyuk = *std::max_element(maxs.begin(), maxs.end());

    uchar tablo[256];
    kontrastTablosu(enKucuk, enBuyuk, tablo);
    return tabloUygula(gri, tablo, threadSayisi);
}

// ---------------------------------------------------------------------------
// Görev 2: histogram + CDF ile histogram eşitleme
// ---------------------------------------------------------------------------
inline void esitlemeTablosu(const long long hist[256], uchar tablo[256]) {
    long long cdf[256];
    cdf[0] = hist[0];
    for (int i = 1; i < 256; ++i) cdf[i] = cdf[i - 1] + hist[i];
    long long N = cdf[255], cdfMin = 0;
    for (int i = 0; i < 256; ++i)
        if (cdf[i] > 0) { cdfMin = cdf[i]; break; }
    for (int v = 0; v < 256; ++v) {
        double oran = N == cdfMin ? 0.0 : double(cdf[v] - cdfMin) / double(N - cdfMin);
        tablo[v] = static_cast<uchar>(std::floor(std::clamp(oran, 0.0, 1.0) * 255.0 + 0.5));
    }
}

inline cv::Mat histogramEsitle(const cv::Mat& gri, int threadSayisi) {
    std::vector<std::vector<long long>> kismi(threadSayisi, std::vector<long long>(256, 0));
    paralel(gri.rows, threadSayisi, [&](int t, int bas, int bit) {
        long long* h = kismi[t].data();
        for (int y = bas; y < bit; ++y) {
            const uchar* s = gri.ptr<uchar>(y);
            for (int x = 0; x < gri.cols; ++x) ++h[s[x]];
        }
    });
    long long hist[256] = {0};
    for (const auto& h : kismi)
        for (int i = 0; i < 256; ++i) hist[i] += h[i];

    uchar tablo[256];
    esitlemeTablosu(hist, tablo);
    return tabloUygula(gri, tablo, threadSayisi);
}

// ---------------------------------------------------------------------------
// Görev 3: CLAHE (OpenCV'nin CPU gerçeklemesiyle aynı sonucu verir)
// ---------------------------------------------------------------------------
struct ClaheBoyut {
    int karoW, karoH, limit;
    float lutOlcek;
};

inline int yansit(int i, int n) { return i < n ? i : 2 * (n - 1) - i; }

inline ClaheBoyut claheBoyut(cv::Size boyut, double kirpmaLimiti, int karoX, int karoY) {
    int genW = boyut.width, genH = boyut.height;
    if (genW % karoX != 0 || genH % karoY != 0) {
        genW += karoX - genW % karoX;
        genH += karoY - genH % karoY;
    }
    ClaheBoyut b;
    b.karoW = genW / karoX;
    b.karoH = genH / karoY;
    int alan = b.karoW * b.karoH;
    b.limit = std::max(static_cast<int>(kirpmaLimiti * alan / 256), 1);
    b.lutOlcek = 255.0f / alan;
    return b;
}

// Kırpılmış histogramdan karo tablosu
inline void claheKaroTablosu(int hist[256], int limit, float lutOlcek, uchar* tablo) {
    int kirpilan = 0;
    for (int i = 0; i < 256; ++i)
        if (hist[i] > limit) {
            kirpilan += hist[i] - limit;
            hist[i] = limit;
        }
    int herSeviyeye = kirpilan / 256, artan = kirpilan - herSeviyeye * 256;
    for (int i = 0; i < 256; ++i) hist[i] += herSeviyeye;
    if (artan > 0) {
        int adim = std::max(256 / artan, 1);
        for (int i = 0; i < 256 && artan > 0; i += adim, --artan) ++hist[i];
    }
    int toplam = 0;
    for (int i = 0; i < 256; ++i) {
        toplam += hist[i];
        tablo[i] = cv::saturate_cast<uchar>(toplam * lutOlcek);
    }
}

inline cv::Mat clahe(const cv::Mat& gri, double kirpmaLimiti, int karoX, int karoY, int threadSayisi) {
    const int W = gri.cols, H = gri.rows;
    ClaheBoyut b = claheBoyut(gri.size(), kirpmaLimiti, karoX, karoY);

    // Karo tabloları: karolar thread'lere paylaştırılır
    std::vector<uchar> tablolar(karoX * karoY * 256);
    paralel(karoX * karoY, threadSayisi, [&](int, int bas, int bit) {
        for (int k = bas; k < bit; ++k) {
            int ky = k / karoX, kx = k % karoX;
            int hist[256] = {0};
            for (int y = ky * b.karoH; y < (ky + 1) * b.karoH; ++y) {
                const uchar* s = gri.ptr<uchar>(yansit(y, H));
                for (int x = kx * b.karoW; x < (kx + 1) * b.karoW; ++x) ++hist[s[yansit(x, W)]];
            }
            claheKaroTablosu(hist, b.limit, b.lutOlcek, &tablolar[k * 256]);
        }
    });

    // Bilineer interpolasyon: satırlar thread'lere paylaştırılır
    cv::Mat cikis(gri.size(), CV_8UC1);
    const float tersW = 1.0f / b.karoW, tersH = 1.0f / b.karoH;
    paralel(H, threadSayisi, [&](int, int bas, int bit) {
        for (int y = bas; y < bit; ++y) {
            float kyf = y * tersH - 0.5f;
            int ky1 = static_cast<int>(std::floor(kyf)), ky2 = ky1 + 1;
            float ya = kyf - ky1, ya1 = 1.0f - ya;
            ky1 = std::max(ky1, 0);
            ky2 = std::min(ky2, karoY - 1);
            const uchar* g = gri.ptr<uchar>(y);
            uchar* c = cikis.ptr<uchar>(y);
            for (int x = 0; x < W; ++x) {
                float kxf = x * tersW - 0.5f;
                int kx1 = static_cast<int>(std::floor(kxf)), kx2 = kx1 + 1;
                float xa = kxf - kx1, xa1 = 1.0f - xa;
                kx1 = std::max(kx1, 0);
                kx2 = std::min(kx2, karoX - 1);
                int p = g[x];
                float ust = tablolar[(ky1 * karoX + kx1) * 256 + p] * xa1 +
                            tablolar[(ky1 * karoX + kx2) * 256 + p] * xa;
                float alt = tablolar[(ky2 * karoX + kx1) * 256 + p] * xa1 +
                            tablolar[(ky2 * karoX + kx2) * 256 + p] * xa;
                c[x] = cv::saturate_cast<uchar>(ust * ya1 + alt * ya);
            }
        }
    });
    return cikis;
}

// ---------------------------------------------------------------------------
// Görev 4: 3x3 ortalama filtre (konvolüsyon, çekirdek = 1/9 * birler matrisi)
// Tam sayı toplam + yuvarlama: round(toplam / 9) = (toplam + 4) / 9
// ---------------------------------------------------------------------------
inline cv::Mat ortalama3x3(const cv::Mat& gri, int threadSayisi) {
    cv::Mat cikis(gri.size(), CV_8UC1);
    paralel(gri.rows, threadSayisi, [&](int, int bas, int bit) {
        for (int y = bas; y < bit; ++y)
            for (int x = 0; x < gri.cols; ++x) {
                int toplam = 0;
                for (int i = -1; i <= 1; ++i) {
                    const uchar* s = gri.ptr<uchar>(std::clamp(y + i, 0, gri.rows - 1));
                    for (int j = -1; j <= 1; ++j) toplam += s[std::clamp(x + j, 0, gri.cols - 1)];
                }
                cikis.at<uchar>(y, x) = static_cast<uchar>((toplam + 4) / 9);
            }
    });
    return cikis;
}

// ---------------------------------------------------------------------------
// Görev 5: 5x5 medyan filtre
// ---------------------------------------------------------------------------
inline cv::Mat medyan5x5(const cv::Mat& gri, int threadSayisi) {
    cv::Mat cikis(gri.size(), CV_8UC1);
    paralel(gri.rows, threadSayisi, [&](int, int bas, int bit) {
        uchar p[25];
        for (int y = bas; y < bit; ++y)
            for (int x = 0; x < gri.cols; ++x) {
                int k = 0;
                for (int i = -2; i <= 2; ++i) {
                    const uchar* s = gri.ptr<uchar>(std::clamp(y + i, 0, gri.rows - 1));
                    for (int j = -2; j <= 2; ++j) p[k++] = s[std::clamp(x + j, 0, gri.cols - 1)];
                }
                for (int a = 1; a < 25; ++a) {  // eklemeli sıralama
                    uchar d = p[a];
                    int b = a - 1;
                    while (b >= 0 && p[b] > d) {
                        p[b + 1] = p[b];
                        --b;
                    }
                    p[b + 1] = d;
                }
                cikis.at<uchar>(y, x) = p[12];
            }
    });
    return cikis;
}

}  // namespace islem


In [ ]:
%%writefile /content/hafta3/odev/main_mt.cpp
// Hafta 3 - Ödev (CPU, multi-threaded)
// Görev 1-5'in çoklu iş parçacıklı sürümleri: her işlem 1, 4 ve tüm
// çekirdek sayısı kadar thread ile çalıştırılıp süreleri karşılaştırılır.
// Tüm thread sayılarında sonucun 1 thread ile birebir aynı olduğu ve
// OpenCV'nin hazır fonksiyonlarına göre farkı kontrol edilir.
//
// Süre ölçümü, farkı belirgin görmek için görüntülerin 2048x2048'e
// büyütülmüş hali üzerinde yapılır. Çıktı resimleri orijinal boyuttadır.
//
// Kullanım: ./odev_mt [goruntu_dizini]   (varsayılan: ../goruntuler)

#include "islemler.hpp"

#include <chrono>
#include <functional>
#include <iostream>
#include <string>

using Saat = std::chrono::steady_clock;
using Islem = std::function<cv::Mat(const cv::Mat&, int)>;

double olcMs(const Islem& f, const cv::Mat& img, int thread, int tekrar) {
    f(img, thread);  // ısınma
    auto bas = Saat::now();
    for (int i = 0; i < tekrar; ++i) f(img, thread);
    return std::chrono::duration<double, std::milli>(Saat::now() - bas).count() / tekrar;
}

int main(int argc, char** argv) {
    std::string dizin = argc >= 2 ? argv[1] : "../goruntuler";
    cv::Mat dusuk = cv::imread(dizin + "/dusuk_kontrast.png", cv::IMREAD_GRAYSCALE);
    cv::Mat gurultulu = cv::imread(dizin + "/gurultulu.png", cv::IMREAD_GRAYSCALE);
    cv::Mat tuzBiber = cv::imread(dizin + "/tuz_biber.png", cv::IMREAD_GRAYSCALE);
    if (dusuk.empty() || gurultulu.empty() || tuzBiber.empty()) {
        std::cerr << "Goruntuler okunamadi (" << dizin << "/dusuk_kontrast.png, gurultulu.png, tuz_biber.png)\n";
        return 1;
    }
    cv::setNumThreads(1);  // OpenCV referans fonksiyonları kendi thread'lerini açmasın

    const int cekirdek = std::max(1u, std::thread::hardware_concurrency());
    std::vector<int> threadSayilari = {1, 4};
    if (cekirdek != 1 && cekirdek != 4) threadSayilari.push_back(cekirdek);

    cv::Ptr<cv::CLAHE> opencvClahe = cv::createCLAHE(2.0, cv::Size(8, 8));
    struct Gorev {
        std::string ad, dosya;
        cv::Mat giris;
        Islem islem;
        std::function<cv::Mat(const cv::Mat&)> opencv;  // karşılaştırma için hazır fonksiyon
    };
    std::vector<Gorev> gorevler = {
        {"1) Kontrast germe", "1_kontrast_gerilmis.png", dusuk, islem::kontrastGer,
         [](const cv::Mat& g) { cv::Mat c; cv::normalize(g, c, 0, 255, cv::NORM_MINMAX); return c; }},
        {"2) Histogram esitleme", "2_esitlenmis.png", dusuk, islem::histogramEsitle,
         [](const cv::Mat& g) { cv::Mat c; cv::equalizeHist(g, c); return c; }},
        {"3) CLAHE", "3_clahe.png", dusuk,
         [](const cv::Mat& g, int t) { return islem::clahe(g, 2.0, 8, 8, t); },
         [&](const cv::Mat& g) { cv::Mat c; opencvClahe->apply(g, c); return c; }},
        {"4) Ortalama 3x3", "4_ortalama.png", gurultulu, islem::ortalama3x3,
         [](const cv::Mat& g) { cv::Mat c; cv::blur(g, c, {3, 3}, {-1, -1}, cv::BORDER_REPLICATE); return c; }},
        {"5) Medyan 5x5", "5_medyan.png", tuzBiber, islem::medyan5x5,
         [](const cv::Mat& g) { cv::Mat c; cv::medianBlur(g, c, 5); return c; }},
    };

    std::cout << "Cekirdek sayisi: " << cekirdek << ", sure olcumu 2048x2048 goruntude\n\n";
    printf("%-24s", "Islem");
    for (int t : threadSayilari) printf(" | %2d thread (ms)", t);
    printf(" | hizlanma | ayni | OpenCV farki\n");

    for (auto& g : gorevler) {
        cv::Mat buyuk;
        cv::resize(g.giris, buyuk, cv::Size(2048, 2048));

        cv::Mat referans = g.islem(buyuk, 1);
        bool hepsiAyni = true;
        for (int t : threadSayilari) hepsiAyni &= cv::norm(referans, g.islem(buyuk, t), cv::NORM_INF) == 0;

        int tekrar = g.ad[0] == '5' ? 2 : 10;
        std::vector<double> sureler;
        for (int t : threadSayilari) sureler.push_back(olcMs(g.islem, buyuk, t, tekrar));

        cv::Mat sonuc = g.islem(g.giris, cekirdek);
        double opencvFark = cv::norm(sonuc, g.opencv(g.giris), cv::NORM_INF);
        cv::imwrite(g.dosya, sonuc);

        printf("%-24s", g.ad.c_str());
        for (double s : sureler) printf(" | %14.2f", s);
        printf(" | %7.2fx | %-4s | %.0f\n", sureler.front() / sureler.back(), hepsiAyni ? "evet" : "HAYIR",
               opencvFark);
    }
    std::cout << "\nhizlanma = 1 thread suresi / en cok thread suresi\n"
                 "ayni     = tum thread sayilarinda sonuc 1 thread ile birebir ayni mi\n"
                 "OpenCV farki = hazir OpenCV fonksiyonuyla en buyuk piksel farki (0 = birebir ayni)\n"
                 "Cikti resimleri bu dizine kaydedildi (1_... 5_...png)\n";
    return 0;
}


In [ ]:
%%writefile /content/hafta3/odev/main_cuda.cu
// Hafta 3 - Ödev (GPU, CUDA)
// Görev 1-5'in CUDA çekirdekleri (kernel) ile gerçeklenmesi. Her işlem için
// GPU sonucu, islemler.hpp'deki çoklu thread CPU sürümüyle karşılaştırılır
// (birebir aynı olmalı) ve süreler ölçülür.
//
//   1) Kontrast germe : paralel min/max indirgeme + tablo uygulama
//   2) Hist. eşitleme : paylaşımlı bellekte blok histogramları + CDF + tablo
//   3) CLAHE          : her karo için bir blok (histogram, kırpma, CDF)
//                       + piksel başına bilineer interpolasyon
//   4) Ortalama 3x3   : piksel başına bir thread
//   5) Medyan 5x5     : piksel başına bir thread, 25 elemanlı sıralama
//
// Süreler: "kernel" = sadece GPU hesaplaması, "toplam" = bellek kopyalama
// (CPU->GPU->CPU) dahil. Ölçüm 2048x2048 görüntüde yapılır.
//
// Derleme (Colab): nvcc -O2 -std=c++17 -fmad=false main_cuda.cu -o odev_cuda \
//                  $(pkg-config --cflags --libs opencv4)
// Kullanım: ./odev_cuda [goruntu_dizini]   (varsayılan: ../goruntuler)

#include "islemler.hpp"

#include <cuda_runtime.h>

#include <chrono>
#include <cstdio>
#include <cstdlib>
#include <functional>
#include <iostream>
#include <string>

#define CUDA_KONTROL(cagri)                                                           \
    do {                                                                              \
        cudaError_t hata = (cagri);                                                   \
        if (hata != cudaSuccess) {                                                    \
            fprintf(stderr, "CUDA hatasi %s:%d: %s\n", __FILE__, __LINE__,            \
                    cudaGetErrorString(hata));                                        \
            exit(1);                                                                  \
        }                                                                             \
    } while (0)

constexpr int BLOK = 256;
constexpr int BLOK_2B = 16;

__device__ __forceinline__ int sinirla(int v, int a, int b) { return v < a ? a : (v > b ? b : v); }
__device__ __forceinline__ unsigned char doyur(float v) {  // cv::saturate_cast<uchar>(float) ile aynı
    return static_cast<unsigned char>(sinirla(__float2int_rn(v), 0, 255));
}

// ---------------------------------------------------------------------------
// Ortak çekirdekler
// ---------------------------------------------------------------------------
__global__ void tabloUygulaK(const unsigned char* giris, unsigned char* cikis, int n,
                             const unsigned char* tablo) {
    __shared__ unsigned char t[256];
    for (int i = threadIdx.x; i < 256; i += blockDim.x) t[i] = tablo[i];
    __syncthreads();
    for (int i = blockIdx.x * blockDim.x + threadIdx.x; i < n; i += gridDim.x * blockDim.x)
        cikis[i] = t[giris[i]];
}

__global__ void histogramK(const unsigned char* giris, int n, unsigned int* hist) {
    __shared__ unsigned int yerel[256];
    for (int i = threadIdx.x; i < 256; i += blockDim.x) yerel[i] = 0;
    __syncthreads();
    for (int i = blockIdx.x * blockDim.x + threadIdx.x; i < n; i += gridDim.x * blockDim.x)
        atomicAdd(&yerel[giris[i]], 1u);
    __syncthreads();
    for (int i = threadIdx.x; i < 256; i += blockDim.x) atomicAdd(&hist[i], yerel[i]);
}

// ---------------------------------------------------------------------------
// Görev 1: kontrast germe
// ---------------------------------------------------------------------------
__global__ void minMaxK(const unsigned char* giris, int n, int* sonuc /* [0]=min, [1]=max */) {
    __shared__ int mn[BLOK], mx[BLOK];
    int yMin = 255, yMax = 0;
    for (int i = blockIdx.x * blockDim.x + threadIdx.x; i < n; i += gridDim.x * blockDim.x) {
        yMin = min(yMin, (int)giris[i]);
        yMax = max(yMax, (int)giris[i]);
    }
    mn[threadIdx.x] = yMin;
    mx[threadIdx.x] = yMax;
    __syncthreads();
    for (int s = blockDim.x / 2; s > 0; s >>= 1) {  // blok içi ağaç indirgeme
        if (threadIdx.x < s) {
            mn[threadIdx.x] = min(mn[threadIdx.x], mn[threadIdx.x + s]);
            mx[threadIdx.x] = max(mx[threadIdx.x], mx[threadIdx.x + s]);
        }
        __syncthreads();
    }
    if (threadIdx.x == 0) {
        atomicMin(&sonuc[0], mn[0]);
        atomicMax(&sonuc[1], mx[0]);
    }
}

__global__ void kontrastTablosuK(const int* minMax, unsigned char* tablo) {
    int v = threadIdx.x;
    int aralik = max(minMax[1] - minMax[0], 1);
    int fark = sinirla(v - minMax[0], 0, aralik);
    tablo[v] = static_cast<unsigned char>((fark * 255 * 2 + aralik) / (2 * aralik));
}

// ---------------------------------------------------------------------------
// Görev 2: histogram eşitleme
// ---------------------------------------------------------------------------
__global__ void esitlemeTablosuK(const unsigned int* hist, unsigned char* tablo) {
    __shared__ long long cdf[256];
    __shared__ long long cdfMin;
    int v = threadIdx.x;
    cdf[v] = hist[v];
    __syncthreads();
    for (int adim = 1; adim < 256; adim <<= 1) {  // Hillis-Steele paralel prefix toplamı
        long long ek = v >= adim ? cdf[v - adim] : 0;
        __syncthreads();
        cdf[v] += ek;
        __syncthreads();
    }
    if (v == 0) {
        cdfMin = 0;
        for (int i = 0; i < 256; ++i)
            if (cdf[i] > 0) { cdfMin = cdf[i]; break; }
    }
    __syncthreads();
    long long N = cdf[255];
    double oran = N == cdfMin ? 0.0 : double(cdf[v] - cdfMin) / double(N - cdfMin);
    oran = fmin(fmax(oran, 0.0), 1.0);
    tablo[v] = static_cast<unsigned char>(floor(oran * 255.0 + 0.5));
}

// ---------------------------------------------------------------------------
// Görev 3: CLAHE
// ---------------------------------------------------------------------------
__device__ __forceinline__ int yansitD(int i, int n) { return i < n ? i : 2 * (n - 1) - i; }

// Her blok bir karo: 256 thread histogramı, kırpmayı ve CDF'yi birlikte yapar
__global__ void claheKaroK(const unsigned char* giris, int W, int H, int karoX, int karoW, int karoH,
                           int limit, float lutOlcek, unsigned char* tablolar) {
    __shared__ int hist[256];
    __shared__ int kirpilan;
    int kx = blockIdx.x % karoX, ky = blockIdx.x / karoX;
    int v = threadIdx.x;
    hist[v] = 0;
    if (v == 0) kirpilan = 0;
    __syncthreads();

    for (int i = v; i < karoW * karoH; i += blockDim.x) {
        int y = yansitD(ky * karoH + i / karoW, H);
        int x = yansitD(kx * karoW + i % karoW, W);
        atomicAdd(&hist[giris[y * W + x]], 1);
    }
    __syncthreads();

    int h = hist[v];
    if (h > limit) {
        atomicAdd(&kirpilan, h - limit);
        h = limit;
    }
    __syncthreads();

    int herSeviyeye = kirpilan / 256, artan = kirpilan - herSeviyeye * 256;
    h += herSeviyeye;
    if (artan > 0) {  // CPU'daki "i += adim" döngüsünün paralel karşılığı
        int adim = max(256 / artan, 1);
        if (v % adim == 0 && v / adim < artan) ++h;
    }
    hist[v] = h;
    __syncthreads();

    if (v == 0) {  // 256 elemanlı CDF: tek thread yeterince hızlı
        int toplam = 0;
        unsigned char* tablo = tablolar + blockIdx.x * 256;
        for (int i = 0; i < 256; ++i) {
            toplam += hist[i];
            tablo[i] = doyur(toplam * lutOlcek);
        }
    }
}

__global__ void claheInterpolasyonK(const unsigned char* giris, unsigned char* cikis, int W, int H,
                                    int karoX, int karoY, float tersW, float tersH,
                                    const unsigned char* tablolar) {
    int x = blockIdx.x * blockDim.x + threadIdx.x;
    int y = blockIdx.y * blockDim.y + threadIdx.y;
    if (x >= W || y >= H) return;

    float kyf = y * tersH - 0.5f;
    int ky1 = static_cast<int>(floorf(kyf)), ky2 = ky1 + 1;
    float ya = kyf - ky1, ya1 = 1.0f - ya;
    ky1 = max(ky1, 0);
    ky2 = min(ky2, karoY - 1);

    float kxf = x * tersW - 0.5f;
    int kx1 = static_cast<int>(floorf(kxf)), kx2 = kx1 + 1;
    float xa = kxf - kx1, xa1 = 1.0f - xa;
    kx1 = max(kx1, 0);
    kx2 = min(kx2, karoX - 1);

    int p = giris[y * W + x];
    float ust = tablolar[(ky1 * karoX + kx1) * 256 + p] * xa1 + tablolar[(ky1 * karoX + kx2) * 256 + p] * xa;
    float alt = tablolar[(ky2 * karoX + kx1) * 256 + p] * xa1 + tablolar[(ky2 * karoX + kx2) * 256 + p] * xa;
    cikis[y * W + x] = doyur(ust * ya1 + alt * ya);
}

// ---------------------------------------------------------------------------
// Görev 4 ve 5: ortalama ve medyan filtre
// ---------------------------------------------------------------------------
__global__ void ortalama3x3K(const unsigned char* giris, unsigned char* cikis, int W, int H) {
    int x = blockIdx.x * blockDim.x + threadIdx.x;
    int y = blockIdx.y * blockDim.y + threadIdx.y;
    if (x >= W || y >= H) return;
    int toplam = 0;
    for (int i = -1; i <= 1; ++i)
        for (int j = -1; j <= 1; ++j) toplam += giris[sinirla(y + i, 0, H - 1) * W + sinirla(x + j, 0, W - 1)];
    cikis[y * W + x] = static_cast<unsigned char>((toplam + 4) / 9);
}

__global__ void medyan5x5K(const unsigned char* giris, unsigned char* cikis, int W, int H) {
    int x = blockIdx.x * blockDim.x + threadIdx.x;
    int y = blockIdx.y * blockDim.y + threadIdx.y;
    if (x >= W || y >= H) return;
    unsigned char p[25];
    int k = 0;
    for (int i = -2; i <= 2; ++i)
        for (int j = -2; j <= 2; ++j) p[k++] = giris[sinirla(y + i, 0, H - 1) * W + sinirla(x + j, 0, W - 1)];
    for (int a = 1; a < 25; ++a) {
        unsigned char d = p[a];
        int b = a - 1;
        while (b >= 0 && p[b] > d) {
            p[b + 1] = p[b];
            --b;
        }
        p[b + 1] = d;
    }
    cikis[y * W + x] = p[12];
}

// ---------------------------------------------------------------------------
// Sunucu (host) tarafı
// ---------------------------------------------------------------------------
struct GpuTampon {
    unsigned char *giris = nullptr, *cikis = nullptr, *tablo = nullptr;
    unsigned int* hist = nullptr;
    int* minMax = nullptr;
    int W = 0, H = 0;

    GpuTampon(int w, int h) : W(w), H(h) {
        CUDA_KONTROL(cudaMalloc(&giris, size_t(W) * H));
        CUDA_KONTROL(cudaMalloc(&cikis, size_t(W) * H));
        CUDA_KONTROL(cudaMalloc(&tablo, 64 * 256));  // CLAHE için 8x8 karo tablosuna yeter
        CUDA_KONTROL(cudaMalloc(&hist, 256 * sizeof(unsigned int)));
        CUDA_KONTROL(cudaMalloc(&minMax, 2 * sizeof(int)));
    }
    ~GpuTampon() {
        cudaFree(giris);
        cudaFree(cikis);
        cudaFree(tablo);
        cudaFree(hist);
        cudaFree(minMax);
    }
};

int izgaraBoyutu(int n) { return std::min((n + BLOK - 1) / BLOK, 1024); }
dim3 izgara2B(int W, int H) { return dim3((W + BLOK_2B - 1) / BLOK_2B, (H + BLOK_2B - 1) / BLOK_2B); }

// Sadece GPU hesaplaması (veri zaten g.giris'te, sonuç g.cikis'e)
void kontrastGerGpu(GpuTampon& g) {
    int n = g.W * g.H, baslangic[2] = {255, 0};
    CUDA_KONTROL(cudaMemcpy(g.minMax, baslangic, sizeof(baslangic), cudaMemcpyHostToDevice));
    minMaxK<<<izgaraBoyutu(n), BLOK>>>(g.giris, n, g.minMax);
    kontrastTablosuK<<<1, 256>>>(g.minMax, g.tablo);
    tabloUygulaK<<<izgaraBoyutu(n), BLOK>>>(g.giris, g.cikis, n, g.tablo);
}

void histogramEsitleGpu(GpuTampon& g) {
    int n = g.W * g.H;
    CUDA_KONTROL(cudaMemset(g.hist, 0, 256 * sizeof(unsigned int)));
    histogramK<<<izgaraBoyutu(n), BLOK>>>(g.giris, n, g.hist);
    esitlemeTablosuK<<<1, 256>>>(g.hist, g.tablo);
    tabloUygulaK<<<izgaraBoyutu(n), BLOK>>>(g.giris, g.cikis, n, g.tablo);
}

void claheGpu(GpuTampon& g) {
    const int karoX = 8, karoY = 8;
    islem::ClaheBoyut b = islem::claheBoyut(cv::Size(g.W, g.H), 2.0, karoX, karoY);
    claheKaroK<<<karoX * karoY, 256>>>(g.giris, g.W, g.H, karoX, b.karoW, b.karoH, b.limit, b.lutOlcek, g.tablo);
    claheInterpolasyonK<<<izgara2B(g.W, g.H), dim3(BLOK_2B, BLOK_2B)>>>(
        g.giris, g.cikis, g.W, g.H, karoX, karoY, 1.0f / b.karoW, 1.0f / b.karoH, g.tablo);
}

void ortalamaGpu(GpuTampon& g) {
    ortalama3x3K<<<izgara2B(g.W, g.H), dim3(BLOK_2B, BLOK_2B)>>>(g.giris, g.cikis, g.W, g.H);
}

void medyanGpu(GpuTampon& g) {
    medyan5x5K<<<izgara2B(g.W, g.H), dim3(BLOK_2B, BLOK_2B)>>>(g.giris, g.cikis, g.W, g.H);
}

// GPU'da çalıştırır; kernel süresini ve kopyalama dahil toplam süreyi (ms) döndürür
cv::Mat gpudaCalistir(const std::function<void(GpuTampon&)>& islem, const cv::Mat& gri, int tekrar,
                      float& kernelMs, float& toplamMs) {
    cv::Mat giris = gri.isContinuous() ? gri : gri.clone();
    cv::Mat cikis(gri.size(), CV_8UC1);
    GpuTampon g(gri.cols, gri.rows);
    size_t boyut = gri.total();

    cudaEvent_t a, b, c, d;
    cudaEventCreate(&a); cudaEventCreate(&b); cudaEventCreate(&c); cudaEventCreate(&d);

    // Isınma
    CUDA_KONTROL(cudaMemcpy(g.giris, giris.data, boyut, cudaMemcpyHostToDevice));
    islem(g);
    CUDA_KONTROL(cudaGetLastError());
    CUDA_KONTROL(cudaDeviceSynchronize());

    kernelMs = toplamMs = 0;
    for (int i = 0; i < tekrar; ++i) {
        cudaEventRecord(a);
        CUDA_KONTROL(cudaMemcpy(g.giris, giris.data, boyut, cudaMemcpyHostToDevice));
        cudaEventRecord(b);
        islem(g);
        cudaEventRecord(c);
        CUDA_KONTROL(cudaMemcpy(cikis.data, g.cikis, boyut, cudaMemcpyDeviceToHost));
        cudaEventRecord(d);
        cudaEventSynchronize(d);
        float k, t;
        cudaEventElapsedTime(&k, b, c);
        cudaEventElapsedTime(&t, a, d);
        kernelMs += k;
        toplamMs += t;
    }
    CUDA_KONTROL(cudaGetLastError());
    kernelMs /= tekrar;
    toplamMs /= tekrar;
    cudaEventDestroy(a); cudaEventDestroy(b); cudaEventDestroy(c); cudaEventDestroy(d);
    return cikis;
}

double cpuOlcMs(const std::function<cv::Mat(const cv::Mat&, int)>& f, const cv::Mat& img, int thread, int tekrar) {
    f(img, thread);
    auto bas = std::chrono::steady_clock::now();
    for (int i = 0; i < tekrar; ++i) f(img, thread);
    return std::chrono::duration<double, std::milli>(std::chrono::steady_clock::now() - bas).count() / tekrar;
}

int main(int argc, char** argv) {
    std::string dizin = argc >= 2 ? argv[1] : "../goruntuler";
    cv::Mat dusuk = cv::imread(dizin + "/dusuk_kontrast.png", cv::IMREAD_GRAYSCALE);
    cv::Mat gurultulu = cv::imread(dizin + "/gurultulu.png", cv::IMREAD_GRAYSCALE);
    cv::Mat tuzBiber = cv::imread(dizin + "/tuz_biber.png", cv::IMREAD_GRAYSCALE);
    if (dusuk.empty() || gurultulu.empty() || tuzBiber.empty()) {
        std::cerr << "Goruntuler okunamadi (" << dizin << "/dusuk_kontrast.png, gurultulu.png, tuz_biber.png)\n";
        return 1;
    }

    int cihazSayisi = 0;
    CUDA_KONTROL(cudaGetDeviceCount(&cihazSayisi));
    if (cihazSayisi == 0) {
        std::cerr << "CUDA destekli GPU bulunamadi.\n";
        return 1;
    }
    cudaDeviceProp ozellik;
    CUDA_KONTROL(cudaGetDeviceProperties(&ozellik, 0));
    const int cekirdek = std::max(1u, std::thread::hardware_concurrency());
    std::cout << "GPU: " << ozellik.name << " | CPU cekirdek: " << cekirdek
              << " | sure olcumu 2048x2048 goruntude\n\n";

    struct Gorev {
        std::string ad, dosya;
        cv::Mat giris;
        std::function<cv::Mat(const cv::Mat&, int)> cpu;
        std::function<void(GpuTampon&)> gpu;
    };
    std::vector<Gorev> gorevler = {
        {"1) Kontrast germe", "1_kontrast_gerilmis_gpu.png", dusuk, islem::kontrastGer, kontrastGerGpu},
        {"2) Histogram esitleme", "2_esitlenmis_gpu.png", dusuk, islem::histogramEsitle, histogramEsitleGpu},
        {"3) CLAHE", "3_clahe_gpu.png", dusuk,
         [](const cv::Mat& g, int t) { return islem::clahe(g, 2.0, 8, 8, t); }, claheGpu},
        {"4) Ortalama 3x3", "4_ortalama_gpu.png", gurultulu, islem::ortalama3x3, ortalamaGpu},
        {"5) Medyan 5x5", "5_medyan_gpu.png", tuzBiber, islem::medyan5x5, medyanGpu},
    };

    printf("%-22s | CPU 1 thr | CPU %2d thr | GPU kernel | GPU toplam | hizlanma* | CPU ile ayni\n",
           "Islem (ms)", cekirdek);
    for (auto& g : gorevler) {
        cv::Mat buyuk;
        cv::resize(g.giris, buyuk, cv::Size(2048, 2048));
        int tekrar = g.ad[0] == '5' ? 2 : 10;

        double cpu1 = cpuOlcMs(g.cpu, buyuk, 1, tekrar);
        double cpuN = cpuOlcMs(g.cpu, buyuk, cekirdek, tekrar);
        float kernelMs, toplamMs;
        cv::Mat gpuSonuc = gpudaCalistir(g.gpu, buyuk, 20, kernelMs, toplamMs);
        double fark = cv::norm(gpuSonuc, g.cpu(buyuk, cekirdek), cv::NORM_INF);

        // Orijinal boyuttaki sonucu kaydet
        float k2, t2;
        cv::imwrite(g.dosya, gpudaCalistir(g.gpu, g.giris, 1, k2, t2));

        printf("%-22s | %9.2f | %10.2f | %10.3f | %10.3f | %8.1fx | %s\n", g.ad.c_str(), cpu1, cpuN, kernelMs,
               toplamMs, cpu1 / kernelMs, fark == 0 ? "evet" : ("HAYIR (fark " + std::to_string(int(fark)) + ")").c_str());
    }
    std::cout << "\n* hizlanma = CPU 1 thread / GPU kernel suresi\n"
                 "GPU toplam = CPU->GPU kopyalama + kernel + GPU->CPU kopyalama\n"
                 "Cikti resimleri bu dizine kaydedildi (*_gpu.png)\n";
    return 0;
}


In [ ]:
%%writefile /content/hafta3/goruntuler/uret.cpp
// Hafta 3 test görüntülerini üretir.
// Verilen (normal) bir fotoğraftan:
//   temiz.png           : gri seviye orijinal
//   dusuk_kontrast.png  : 0.35 * p + 80  (değerler yaklaşık 80..170 arasına sıkışır)
//   gurultulu.png       : Gauss gürültüsü eklenmiş (sigma = 20)
//   tuz_biber.png       : %5 tuz (255) + %5 biber (0) gürültüsü
//
// Kullanım: ./uret <kaynak_fotograf>   (çıktılar çalışılan dizine yazılır)

#include <opencv2/opencv.hpp>

#include <iostream>

int main(int argc, char** argv) {
    if (argc < 2) {
        std::cerr << "Kullanim: " << argv[0] << " <kaynak_fotograf>\n";
        return 1;
    }
    cv::Mat gri = cv::imread(argv[1], cv::IMREAD_GRAYSCALE);
    if (gri.empty()) {
        std::cerr << "Goruntu okunamadi: " << argv[1] << "\n";
        return 1;
    }
    cv::theRNG().state = 42;  // her seferinde aynı gürültü

    cv::Mat dusuk;
    gri.convertTo(dusuk, CV_8U, 0.35, 80);

    cv::Mat gurultu(gri.size(), CV_16S), gurultulu;
    cv::randn(gurultu, 0, 20);
    cv::Mat gri16;
    gri.convertTo(gri16, CV_16S);
    cv::Mat(gri16 + gurultu).convertTo(gurultulu, CV_8U);

    cv::Mat tuzBiber = gri.clone(), rastgele(gri.size(), CV_32F);
    cv::randu(rastgele, 0, 1);
    tuzBiber.setTo(0, rastgele < 0.05);
    tuzBiber.setTo(255, rastgele > 0.95);

    cv::imwrite("temiz.png", gri);
    cv::imwrite("dusuk_kontrast.png", dusuk);
    cv::imwrite("gurultulu.png", gurultulu);
    cv::imwrite("tuz_biber.png", tuzBiber);
    std::cout << "Uretildi (" << gri.cols << "x" << gri.rows
              << "): temiz.png, dusuk_kontrast.png, gurultulu.png, tuz_biber.png\n";
    return 0;
}


## 3. Test görüntüleri
Bir kaynak fotoğraftan düşük kontrastlı, Gauss gürültülü ve tuz-biber gürültülü görüntüler üretilir. Kendi fotoğrafınızı `kaynak.jpg` olarak yükleyebilirsiniz.

In [ ]:
%cd /content/hafta3/goruntuler
import os
if not os.path.exists("kaynak.jpg"):
    !wget -q -O kaynak.jpg https://raw.githubusercontent.com/opencv/opencv/4.x/samples/data/lena.jpg
!g++ -O2 -std=c++17 uret.cpp -o uret $(pkg-config --cflags --libs opencv4) && ./uret kaynak.jpg

## 4. Derleme

In [ ]:
%cd /content/hafta3/odev
!g++ -O2 -std=c++17 -ffp-contract=off -pthread main_mt.cpp -o odev_mt $(pkg-config --cflags --libs opencv4)
!nvcc -O2 -std=c++17 -arch=native -fmad=false -Xcompiler -ffp-contract=off main_cuda.cu -o odev_cuda $(pkg-config --cflags --libs opencv4)
!ls -la odev_mt odev_cuda

## 5. Çoklu thread (CPU) sürümü

In [ ]:
!./odev_mt

## 6. CUDA (GPU) sürümü

In [ ]:
!./odev_cuda

## 7. Sonuçlar

In [ ]:
import cv2
from google.colab.patches import cv2_imshow
for ad in ["1_kontrast_gerilmis", "2_esitlenmis", "3_clahe", "4_ortalama", "5_medyan"]:
    cpu = cv2.imread(f"{ad}.png", 0)
    gpu = cv2.imread(f"{ad}_gpu.png", 0)
    print(ad, "(sol: CPU, sag: GPU)")
    cv2_imshow(cv2.hconcat([cv2.resize(cpu, (384, 384)), cv2.resize(gpu, (384, 384))]))